In [ ]:
# import libraries
import pandas as pd
import ee
import geemap

## Connect to GOOGLE EARTH ENGINE

In [ ]:
# Authenticate GEE
ee.Authenticate()

In [ ]:
# Initialize GEE
ee.Initialize()

print("Google Earth Engine initialized successfully!")

In [ ]:
# Center coordinates to show map
fct_center =  (9.056266, 7.498522)

## Visualise Parameters

In [ ]:
# Boundary visualization params 
vis_params_fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}

vis_params_aoi = {"fillcolor": "", "color": "red"}

# Sentinel-2 Visualization parameters 
vis_params_s2_rgb = {"min" : 300, "max" :3000, "bands": ["B4", "B3", "B2"]}
vis_params_s2_fcc = {"min" : 300, "max" :3000, "bands": ["B8", "B4", "B3"]}


# Spectral indices visualization parameters
# NDVI
ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": [
        "#a50026",  
        "#d73027",
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#66bd63",
        "#1a9850",
        "#006837",  
    ],
}

# NDBI
ndbi_vis = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "#f7f7f7",  
        "#fddbc7",
        "#f4a582",
        "#d6604d",
        "#b2182b",  
    ],
}

# NDWI
ndwi_vis = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "#543005",  
        "#8c510a",
        "#d8b365",
        "#f6e8c3",
        "#c7eae5",
        "#5ab4ac",
        "#01665e",  
    ],
}


# Visualisation parameters for road layers
vis_params_roads_vector = {
                        "color": "red",
                        "width": 1.5,
                    }

vis_params_roads_raster = {
                        "min": 0,
                        "max": 1,
                        "palette": ["black", "white"],
                    }


vis_params_dist_road = {
    "min": 0,
    "max": 21730, #meters
    "palette": [
        "#d73027",  
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#1a9850",  
    ],
}

# Visualisation parameters for water layers
vis_params_water = {
    "min": 0,
    "max": 1,
    "palette": ["white", "blue"], 
}

# Distance to water
vis_params_dist_water = {
    "min": 0,
    "max": 38497,  
    "palette": [
        "#f7fbff",  
        "#deebf7",
        "#9ecae1",
        "#4292c6",
        "#2171b5",
        "#08306b",  
    ],
}


# Nighttime lights visualisation parameters
vis_params_ntl = {
    "min": 0,
    "max": 20,  
    "palette": [
        "#000000",  
        "#2c0b00",
        "#6e1c00",
        "#a83800",
        "#d9720a",
        "#f7b733",
        "#ffe98a",  
    ],
}


# Visualization parameters for GPWv411 population DENSITY layer
vis_params_gpw = {
  "min": 0.0,
  "max": 10000.0,
  "palette": ["ffffe7", "FFc869", "ffac1d", "e17735", "f2552c", "9f0c21"]
  }

## Filter Boundaries & Set AOI

In [ ]:
# Boundary Data From FAO GAUL
# Data source: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_SIMPLIFIED_500m_2015_level0

# 1. Load FAO GAUL Administrative Boundaries
fao_gaul_l0 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0')
fao_gaul_l1 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1')
fao_gaul_l2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2')

# 2. Filter to Nigeria
nigeria_country = fao_gaul_l0.filter(ee.Filter.eq('ADM0_NAME', 'Nigeria'))
nigeria_states  = fao_gaul_l1.filter(ee.Filter.eq('ADM0_NAME', 'Nigeria'))
nigeria_lgas    = fao_gaul_l2.filter(ee.Filter.eq('ADM0_NAME', 'Nigeria'))

# 3. Define Standardized Hollow Vector Styles (Matching Demo Repository Symbology)
style_country = {"color": "000000", "width": 2.0, "fillColor": "00000000"}
style_states  = {"color": "00909F", "width": 1.2, "fillColor": "00000000"}
style_lgas    = {"color": "E67E22", "width": 0.7, "fillColor": "00000000"}

## Filter to Nigeria

In [ ]:
# Initialize interactive map centered on Nigeria / FCT coordinates
m = geemap.Map(center=[9.056266, 7.498522], zoom=7)

# Add boundary layers using styled feature collections
m.addLayer(nigeria_country.style(**style_country), {}, 'Nigeria Boundary')
m.addLayer(nigeria_states.style(**style_states), {}, 'State Boundaries')
m.addLayer(nigeria_lgas.style(**style_lgas), {}, 'LGA Boundaries')

# Add layer control and display map
m.add_layer_control()
m

In [ ]:
# Select just a single feature
print(fao_gaul_l0.limit(1).getInfo()["columns"])
nga_l0 = fao_gaul_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
nga_l1 = fao_gaul_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
fct_l0 = nga_l1.filter(ee.Filter.eq("ADM1_NAME", "Abuja"))
print(nga_l0.getInfo())

#get geometry of Abuja Boundary Feature Collection
aoi = fct_l0.geometry()
aoi_bbox = aoi.bounds()

#Create a map to visualize Abuja Boundary
aoi_map = geemap.Map(center=(7.0, 8.0), zoom=10)

# 7. Add layer (Fixed capital 'L' in addLayer)
aoi_map.addLayer(fct_l0, vis_params_aoi, 'Abuja Boundary')
aoi_map


## Explore image operations
### AOI, SCL Cloud Masking, & Median Composite

In [ ]:
# Cloud masking function using SCL (Scene Classification Layer)
def mask_s2_clouds(image):
    scl = image.select('SCL')
    # Keep clear land (4), vegetation (5), water (6), unclassified (7)
    mask = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
    return image.updateMask(mask)

# Download and process Sentinel-2 collection over Abuja
def get_sentinel2_composite(aoi, start_date='2022-01-01', end_date='2022-01-31'):
    s2_img_col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')   # All Sentinel-2 collection
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_s2_clouds)
    )
    return s2_img_col

# Call the function to create s2_img_col
s2_img_col = get_sentinel2_composite(aoi)

# Collection Properties
print(f"Number of images in S2 collection: {s2_img_col.size().getInfo()}\n")

# Get the first image in the collection and print its properties
first_s2_img = s2_img_col.first()
print(f"First image in S2 collection: {first_s2_img.getInfo()}\n")

# Bands in S2 image
print(f"Bands in S2 image: {first_s2_img.bandNames().getInfo()}")

# Select just "B4"
red_band_s2 = first_s2_img.select('B4')
print(f"Red band (B4) in S2 image: {red_band_s2.bandNames().getInfo()}")

# Visualise s2 image
vis_params_s2 = {
    'bands': ['B8', 'B4', 'B3'],  # RGB bands
    'min': 300,
    'max': 3000,
    'gamma': 1.4
}
s2_map = geemap.Map(center=(7.13, 8.84), zoom=8)
s2_map.addLayer(first_s2_img, vis_params_s2, 'First S2 Image')
s2_map

In [ ]:
# Mosaic the entire S2 collection & clip to FCT extent [Spatial Mosaic]
s2_mosaic = s2_img_col.mosaic()
s2_mosaic_clipped = s2_mosaic.clip(aoi)
print(f"Mosaiced S2 Collection {s2_mosaic.getInfo()}")

# Create a fresh map instance for the mosaic view
s2_mosaic_map = geemap.Map(center=[9.0765, 7.3986], zoom=9)

# Add layer to the new map
s2_mosaic_map.addLayer(s2_mosaic_clipped, vis_params_s2_rgb, "S2 Mosaiced - Spatial")

# Display the new map
s2_mosaic_map

In [ ]:
# Median of the S2 collection
s2_img_median = s2_img_col.median()
print(f"Median of S2 collection: {s2_img_median.getInfo()}")

# Visualise s2 image
vis_params_s2 = {
    'bands': ['B8', 'B4', 'B3'],  # RGB bands
    'min': 300,
    'max': 3000,
    'gamma': 1.4
}
s2_median_map = geemap.Map(center=(7.13, 8.84), zoom=8)
s2_median_map.addLayer(s2_img_median, vis_params_s2, 'S2 Median Image')
s2_median_map

### SPECTRAL INDICES COMPUTATION FOR URBAN GROWTH & LAND COVER ANALYSIS

In [ ]:
# Assumes 's2_img_median' is your cloud-masked, median composite Sentinel-2 SR image clipped to your AOI
# Sentinel-2 Bands: B8 = NIR | B4 = Red | B3 = Green | B11 = SWIR 1

# 1. Calculate Indices
ndvi  = s2_img_median.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndbi  = s2_img_median.normalizedDifference(['B11', 'B8']).rename('NDBI')
ndwi  = s2_img_median.normalizedDifference(['B3', 'B8']).rename('NDWI')
mndwi = s2_img_median.normalizedDifference(['B3', 'B11']).rename('MNDWI')

# 2. Add Indices as Bands to the Sentinel-2 Image Composite
s2_with_indices = s2_img_median.addBands([ndvi, ndbi, ndwi, mndwi])

# 3. Add Layers to Map using existing vis parameters defined above
m.addLayer(ndvi, ndvi_vis, 'NDVI (Vegetation)')
m.addLayer(ndbi, ndbi_vis, 'NDBI (Built-up)')
m.addLayer(ndwi, ndwi_vis, 'NDWI (Water)')

# Re-render layer control
m.add_layer_control()
m

## Land Cover / Existing Urban Layer

### Drivers of Urban Growth / Predictors

#### Elevation/DEM

In [ ]:
# Download elevation and compute slope
dem = ee.Image('USGS/SRTMGL1_003')
dem.bandNames().getInfo()



elevation = dem.select('elevation')
slope = ee.Terrain.slope(elevation)


dem_map = geemap.Map(center = [7.08, 8.88], zoom=8)
dem_map.add_basemap("SATELLITE")
dem_map.addLayer(dem.clip(aoi), {'min': 0, 'max': 500, "palette": ["Red", "Green", "Yellow"]}, "DEM")
dem_map.addLayer(slope.clip(aoi), {'min': 0, 'max': 10, "palette": ["Red", "Green", "Yellow"]}, "Slope")
dem_map

In [ ]:
m = geemap.Map()
m.set_center(-112.8598, 36.2841, 10)
m.add_layer(slope, {'min': 0, 'max': 60}, 'slope')
m

## Land Cover / Existing Urban Layer (GLC_FCS30D: 2012–2022)
The raw GLC_FCS30D asset is stored as tiled, multi-band images (one band per year). We mosaic the tiles, rename the bands to actual years, and convert the multi-band image into a proper year-indexed ImageCollection, following the official preprocessing pattern published by the data provider.

In the GLC_FCS30D legend, class code 190 = Impervious surfaces (built-up / urban). We use this to derive a binary urban mask for every year. 

In [ ]:
# GLC_FCS30D Global Land Cover
# Data source : https://gee-community-catalog.org/projects/glc_fcs/?h=glc+fcs30d
# Reference   : https://gee-community-catalog.org/tutorials/examples/glc_fcs30d_lulc/

# Annual land cover ImageCollection (2000 – 2022). 
# Each image in annual land cover data has 23 bands, one for each year from 2000-2022 (23 years)
# (bands b1, b2,..b23 = 2000, 2001,..2022)
glc_annual = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")

# Classification scheme
# (35 landcover class and 1 fill value)
glc_class_values =  [
  10, 11, 12, 20, 51, 52, 61, 62, 71, 72, 81, 82, 91, 92, 120, 121, 122, 
  130, 140, 150, 152, 153, 181, 182, 183, 184, 185, 186, 187, 190, 200, 
  201, 202, 210, 220, 0
]

# Land cover class names
glc_class_names = [
    "Rainfed_cropland", "Herbaceous_cover_cropland", "Tree_or_shrub_cover_cropland",
    "Irrigated_cropland", "Open_evergreen_broadleaved_forest", "Closed_evergreen_broadleaved_forest",
    "Open_deciduous_broadleaved_forest", "Closed_deciduous_broadleaved_forest",
    "Open_evergreen_needle_leaved_forest", "Closed_evergreen_needle_leaved_forest",
    "Open_deciduous_needle_leaved_forest", "Closed_deciduous_needle_leaved_forest",
    "Open_mixed_leaf_forest", "Closed_mixed_leaf_forest", "Shrubland",
    "Evergreen_shrubland", "Deciduous_shrubland", "Grassland", "Lichens_and_mosses",
    "Sparse_vegetation", "Sparse_shrubland", "Sparse_herbaceous", "Swamp", "Marsh",
    "Flooded_flat", "Saline", "Mangrove", "Salt_marsh", "Tidal_flat",
    "Impervious_surfaces", "Bare_areas", "Consolidated_bare_areas",
    "Unconsolidated_bare_areas", "Water_body", "Permanent_ice_and_snow", "Filled_value",
]

glc_class_colours = [
    "#ffff64", "#ffff64", "#ffff00", "#aaf0f0", "#4c7300",
    "#006400", "#a8c800", "#00a000", "#005000", "#003c00",
    "#286400", "#285000", "#a0b432", "#788200", "#966400",
    "#964b00", "#966400", "#ffb432", "#ffdcd2", "#ffebaf",
    "#ffd278", "#ffebaf", "#00a884", "#73ffdf", "#9ebb3b",
    "#828282", "#f57ab6", "#66cdab", "#444f89", "#c31400",
    "#fff5d7", "#dcdcdc", "#fff5d7", "#0046c8", "#ffffff", "#ffffff",
]


In [ ]:
# Mosaic tiled images and rename bands b1, b2, ... to 2000, 2001, ...
glc_mosaic = glc_annual.mosaic()
years_list = ee.List.sequence(2000, 2022).map(lambda year: ee.Number(year).format("%04d"))
glc_mosaic_renamed = glc_mosaic.rename(years_list)

# Multiband to single-band annual mosaic & assign time and year metadata to each 
glc_mosaic_renamed_upd = years_list.map(
    lambda year: glc_mosaic_renamed
    .select([year])
    .set({
        "system:time_start": ee.Date.fromYMD(ee.Number.parse(year), 1, 1).millis(),
        "system:index": year,
        "year": ee.Number.parse(year),
    })
)

glc_mosaics_col = ee.ImageCollection.fromImages(glc_mosaic_renamed_upd)
print(glc_mosaics_col.first().getInfo())

# Remap native class values to sequential integers
new_class_values = ee.List.sequence(1, ee.List(glc_class_values).length())

## Drivers of Urban Growth / Predictors

In [ ]:
# Create maps to visualise thematic layers
thematic_map_1 = geemap.Map(center = fct_center, zoom=8)
thematic_map_1.add_basemap("SATELLITE")

thematic_map_2 = geemap.Map(center = fct_center, zoom=8)
thematic_map_2.add_basemap("SATELLITE")

### Elevation/DEM

In [ ]:
# Download elevation and compute slope
# Data source: https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003
dem = ee.Image('USGS/SRTMGL1_003')
print(f"Bands in the DEM {dem.bandNames().getInfo()}")


elevation = dem.select('elevation')
slope = ee.Terrain.slope(elevation)



thematic_map_1.addLayer(dem.clip(aoi), {'min': 0, 'max': 500, "palette": ["Red", "Green", "Yellow"]}, "DEM")
thematic_map_1.addLayer(slope.clip(aoi), {'min': 0, 'max': 10, "palette": ["Red", "Green", "Yellow"]}, "Slope")
thematic_map_1

### Distance to Road

In [ ]:
# Distance to roads
# Data source: https://gee-community-catalog.org/projects/grip/
roads_africa = ee.FeatureCollection("projects/sat-io/open-datasets/GRIP4/Africa")

# Roads that intersect with study extent
roads_aoi = roads_africa.filterBounds(aoi_bbox)

# Convert road from vector to raster & compute eucledian distance 
roads_raster = ee.Image().float().paint(roads_aoi, 1).clip(aoi)
distance_to_roads = (
    roads_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())  # convert pixel distance to meters
    .rename("dist_to_roads")
    .clip(aoi)
)

# Maximum distance in 'distance_to_roads' layer
print(distance_to_roads.reduceRegion(ee.Reducer.max(), aoi, 1000, maxPixels=1e9).getInfo())

thematic_map_1.addLayer(roads_raster, vis_params_roads_raster, "Road Raster")
thematic_map_1.addLayer(distance_to_roads.select("dist_to_roads"), vis_params_dist_road, "Distance to Road")
thematic_map_1.addLayer(roads_aoi, vis_params_roads_vector, "Road Vector")
thematic_map_1

### Distance to Permanent Water

In [ ]:
# Permanent water (JRC Global Surface Water)
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").clip(aoi)
permanent_water = gsw.select("occurrence").gte(50)  # >=50% of the time = water

# compute eucledian distance 
distance_to_water = (
    permanent_water.Not()
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .rename("dist_to_water")
    .clip(aoi)
)

# 
print(distance_to_water.reduceRegion(ee.Reducer.max(), aoi, 1000, maxPixels=1e9).getInfo())

thematic_map_1.addLayer(permanent_water, vis_params_water, "Permanent Water")
thematic_map_1.addLayer(distance_to_water.select("dist_to_water"), vis_params_dist_water, "Distance to Water")
thematic_map_1

### Nighttime Lights

In [ ]:
date = ee.Date("2015-01-01")
date_plus_1year = date.advance(-1, "year")

print(date.format("YYYY-MM-dd").getInfo())
print(date_plus_1year.format("YYYY-MM-dd").getInfo())

In [ ]:
# Nighttime lights (VIIRS DNB, annual composite)
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG
def get_nighttime_lights(year, study_extent):
    '''Annual mean VIIRS radiance composite.'''
    start = ee.Date.fromYMD(year, 1, 1) # "2015", "january", "1"
    end = start.advance(1, "year")
    composite = (
        ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
        .filterDate(start, end)
        .select("avg_rad")
        .mean()
        .rename("ntl")
        .clip(study_extent)
    )
    return composite


# Apply function 
ntl_2015 = get_nighttime_lights(2015, aoi)
ntl_2020 = get_nighttime_lights(2020, aoi)
ntl_2025 = get_nighttime_lights(2025, aoi)


thematic_map_2.addLayer(ntl_2015, vis_params_ntl, "NTL - 2015")
thematic_map_2.addLayer(ntl_2020, vis_params_ntl, "NTL - 2020")
thematic_map_2.addLayer(ntl_2025, vis_params_ntl, "NTL - 2025")
thematic_map_2

### Population Density

In [ ]:
# Population density (CIESIN GPWv4.11) 
# Data source: https://developers.google.com/earth-engine/datasets/catalog/CIESIN_GPWv411_GPW_Population_Density
# Closest available year to analysis baseline (2015 / 2020 / 2022).
gpw = ee.ImageCollection("CIESIN/GPWv411/GPW_Population_Density")

def get_population_density(year, extent):
    '''Return the GPWv4.11 population density image closest to the given year.'''
    available_years = [2000, 2005, 2010, 2015, 2020]
    closest_year = min(available_years, key=lambda y: abs(y - year))
    image = (
        gpw.filter(ee.Filter.calendarRange(closest_year, closest_year, "year"))
        .first()
        .select("population_density")
        .rename("pop_density")
        .clip(extent)
    )
    return image

# Apply function
pop_density_2015 = get_population_density(2015, aoi)
pop_density_2020 = get_population_density(2020, aoi)
pop_density_2022 = get_population_density(2022, aoi)   


thematic_map_2.addLayer(pop_density_2015, vis_params_gpw, "Population Density - 2015")
thematic_map_2.addLayer(pop_density_2020, vis_params_gpw, "Population Density - 2020")
thematic_map_2.addLayer(pop_density_2022, vis_params_gpw, "Population Density - 2022 (uses 2020 data)")

thematic_map_2

### Building Density

In [ ]:
m = geemap.Map()
m.set_center(-112.8598, 36.2841, 10)
m.add_layer(slope, {'min': 0, 'max': 60}, 'slope')
m

### Spectral Indices

In [ ]:
# Spectral indices from Landsat (annual cloud-masked median composite) 
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2
def mask_landsat_clouds(image):
    qa = image.select("QA_PIXEL")
    cloud_bit, shadow_bit = 1 << 3, 1 << 4
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(shadow_bit).eq(0))
    return image.updateMask(mask)

def get_spectral_indices(year):
    '''NDVI, NDBI, NDWI annual median composite from Landsat 8/9 SR.'''
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(start, end)
    l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(aoi).filterDate(start, end)
    landsat = l8.merge(l9).map(mask_landsat_clouds)

    # Scale factors for Collection 2 Level-2 surface reflectance
    def apply_scale_factors(image):
        optical = image.select("SR_B.").multiply(0.0000275).add(-0.2)
        return image.addBands(optical, None, True)

    landsat = landsat.map(apply_scale_factors)
    composite = landsat.median().clip(aoi)

    ndvi = composite.normalizedDifference(["SR_B5", "SR_B4"]).rename("ndvi")
    ndbi = composite.normalizedDifference(["SR_B6", "SR_B5"]).rename("ndbi")
    ndwi = composite.normalizedDifference(["SR_B3", "SR_B5"]).rename("ndwi")

    return ee.Image.cat([ndvi, ndbi, ndwi])

indices_2015 = get_spectral_indices(2015)
indices_2020 = get_spectral_indices(2020)
indices_2025 = get_spectral_indices(2025)


# Add layers to map 
for year, indices in [(2015, indices_2015), (2020, indices_2020), (2025, indices_2025)]:
    thematic_map_2.addLayer(indices.select("ndvi"), ndvi_vis, f"NDVI - {year}")
    thematic_map_2.addLayer(indices.select("ndbi"), ndbi_vis, f"NDBI - {year}")
    thematic_map_2.addLayer(indices.select("ndwi"), ndwi_vis, f"NDWI - {year}")

thematic_map_2

### 1. BUILDINGS FEATURE: GOOGLE OPEN BUILDINGS DENSITY LAYER

In [ ]:
# Load Google Open Buildings dataset for Africa
open_buildings = ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons").filterBounds(aoi)

# Paint building polygons to raster (1 where building exists, 0 elsewhere)
building_raster = ee.Image().float().paint(open_buildings, 1).clip(aoi).unmask(0).rename("building_presence")

# Calculate localized building density (focal mean within a 150m radius kernel)
building_density = building_raster.reduceNeighborhood(
    reducer=ee.Reducer.mean(),
    kernel=ee.Kernel.circle(radius=150, units='meters')
).rename("building_density").clip(aoi)

### 2. FEATURE STACK ASSEMBLY (training_image)

In [ ]:
# Combine class labels (GLC_FCS30D) and all predictor variables into one image
# Note: Ensure variables (elevation, slope, distance_to_roads, etc.) match your notebook definitions
training_image = ee.Image.cat([
    glc_mosaics_col.first().select([0], ['landcover']), # Land cover class target
    elevation,
    slope,
    distance_to_roads,
    distance_to_water,
    ntl_2015,
    pop_density_2015,
    indices_2015,
    building_density  # Building footprint density layer included
]).clip(aoi)


### 3. STRATIFIED SAMPLING FUNCTION DEFINITION

In [ ]:
def generate_stratified_samples(image, class_band, region, num_points=500, scale=30, seed=42):
    """
    Generates stratified random sample points across land cover classes 
    and extracts pixel values from input feature bands.
    
    Parameters:
    - image (ee.Image): Feature image composite containing target class and predictors.
    - class_band (str): Name of the class label band inside the image.
    - region (ee.Geometry): Area of interest (AOI) bounding box.
    - num_points (int): Points per land cover class (Default: 500).
    - scale (int): Sampling spatial resolution in meters (Default: 30).
    - seed (int): Seed for deterministic point generation.
    
    Returns:
    - ee.FeatureCollection: Sampled points containing extracted pixel properties.
    """
    samples = image.stratifiedSample(
        numPoints=num_points,
        classBand=class_band,
        region=region,
        scale=scale,
        seed=seed,
        geometries=True  # Retains spatial point geometries for visualization/export
    )
    return samples

### 4. FUNCTION EXECUTION & VALIDATION

In [ ]:
# Generate 500 points per class from the feature stack
training_samples = generate_stratified_samples(
    image=training_image,
    class_band='landcover',
    region=aoi,
    num_points=500,
    scale=30
)

# Print execution results to verify
print("Total sample points generated:", training_samples.size().getInfo())
print("First sample point properties:", training_samples.first().getInfo())